# Notebook 1 — Baseline LLM Fact-Checking: Closed-Book vs. Open-Book

**Goal:** measure how well an LLM fact-checks real PolitiFact claims under two
conditions:

1. **Closed-book (parametric memory only).** The model sees the claim and
   nothing else — no tools. Whatever it "knows" was frozen into its weights
   at training time.
2. **Open-book (web search).** The same model, same claim, but with the
   OpenAI Responses API's native `web_search` tool attached, so it can look
   up live evidence before answering.

## Why this distinction matters

An LLM's parametric memory is a lossy, dated compression of its training
corpus: closed-book fact-checking tests *recall under uncertainty*. Attaching
web search instead tests *retrieval plus reasoning over whatever comes back* —
and what comes back is not curated. Matthew DeVerna's ACL 2026 findings show
that **uncurated search results and reasoning chains frequently fail to
confirm complex political claims** — search access is not a guarantee of
better verdicts, which is exactly why we measure both conditions instead of
assuming.

## The data and the tooling

- **Real PolitiFact fact-checks** (scraped 2024-10-10; 24,964 claims), copied
  from the `llm-vs-human-fc-agreement` project. Each row has the claim
  (`statement`), who said it (`statement_originator`), and PolitiFact's
  Truth-O-Meter verdict.
- The local **`toolkit`** package (installed editable via `uv sync`) carries
  the methodology from that project: the 7-label structured-output schema
  (`OpenAIFactCheckingResponse` — six Truth-O-Meter labels plus *"Not enough
  information"*), the prompts, the prompt builder, and label-collapse maps
  for scoring at **multi-class / ternary / binary** granularity.


In [ ]:
import os
from pathlib import Path

# Notebooks live in notebooks/; run from the repo root so relative paths like
# ./data behave identically in VS Code and classic Jupyter. The local
# `toolkit` package is installed (editable) in the .venv, so no sys.path hacks.
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
os.chdir(repo_root)
print(f"Working directory: {Path.cwd()}")

import pandas as pd

from toolkit.config import DEFAULT_LLM, FACTCHECK_DATA_PATH
from toolkit.metrics import NEI, SCENARIOS, VERACITY_VERDICTS
from toolkit.prompts import CLOSED_BOOK_PROMPT, SIMPLE_PROMPT
from toolkit.providers import openai_provider
from toolkit.response_structure import OpenAIFactCheckingResponse
from toolkit.text import build_user_text

print(f"Model: {DEFAULT_LLM}")
print(f"Closed-book prompt: {CLOSED_BOOK_PROMPT}")
print(f"Open-book prompt:   {SIMPLE_PROMPT}")


## 1. Load the PolitiFact data

PolitiFact also publishes Flip-O-Meter ratings ("Full flop", "Half flip",
"No flip") that rate position *changes*, not factuality — we filter to the six
Truth-O-Meter verdicts (`toolkit.metrics.VERACITY_VERDICTS`). We then draw a
small seeded sample so the workshop run is fast, cheap, and reproducible.
Notebook 5 draws the **same seeded sample**, so results join cleanly.


In [ ]:
RANDOM_SEED = 303
N_CLAIMS = 10

claims_df = pd.read_parquet(FACTCHECK_DATA_PATH)
print(f"Loaded {len(claims_df):,} fact-checks")

claims_df = claims_df[claims_df["verdict"].isin(VERACITY_VERDICTS)]
print(f"{len(claims_df):,} remain after filtering to Truth-O-Meter verdicts")

sample_df = claims_df.sample(n=N_CLAIMS, random_state=RANDOM_SEED).reset_index(drop=True)
sample_df[["statement", "statement_originator", "statement_date", "verdict"]]


## 2. Structured verdicts through the toolkit provider

Free-text verdicts are unusable at scale, so every call goes through
`toolkit.providers.openai_provider.run_parsed`, which wraps the OpenAI
Responses API's `client.responses.parse(...)`:

- output is constrained to `OpenAIFactCheckingResponse` — a Pydantic model
  whose `label` field is a `Literal` over the six Truth-O-Meter labels plus
  `"Not enough information"`;
- `use_web_search=True` attaches the native `web_search` tool (open-book);
  `False` sends the bare claim (closed-book);
- transient API errors are retried with exponential backoff (`tenacity`).

The user prompt is built by `toolkit.text.build_user_text(originator,
statement)` — the exact formatting used in the source project.


In [ ]:
print(OpenAIFactCheckingResponse.model_json_schema()["properties"]["label"]["enum"])


def fact_check(row, open_book):
    system_prompt = SIMPLE_PROMPT if open_book else CLOSED_BOOK_PROMPT
    user_text = build_user_text(row.statement_originator, row.statement)
    try:
        parsed, _raw = openai_provider.run_parsed(
            DEFAULT_LLM, system_prompt, user_text, use_web_search=open_book
        )
        return parsed
    except Exception as e:
        print(f"  ! API error for claim '{row.statement[:50]}...': {e}")
        return None


## 3. Condition A — Closed-book


In [ ]:
closed_book = []
for row in sample_df.itertuples():
    verdict = fact_check(row, open_book=False)
    closed_book.append(verdict)
    label = verdict.label if verdict else "ERROR"
    print(f"[closed-book] gold={row.verdict:<13} pred={label:<22} {row.statement[:55]}...")


## 4. Condition B — Open-book (web search)

Same model, same claims, but the request now carries the `web_search` tool
and the system prompt *requires* the model to search before answering. These
calls take noticeably longer — each one performs live retrieval.


In [ ]:
open_book = []
for row in sample_df.itertuples():
    verdict = fact_check(row, open_book=True)
    open_book.append(verdict)
    label = verdict.label if verdict else "ERROR"
    print(f"[open-book]   gold={row.verdict:<13} pred={label:<22} {row.statement[:55]}...")


## 5. Scoring at three granularities

Exact 6-way agreement with PolitiFact is a harsh metric — "Mostly false" vs.
"False" is a rating-scale quibble, not a fact-checking failure. So we score
with the collapse maps from `toolkit.metrics.SCENARIOS`, exactly as in the
source project:

- **multi_class** — exact label match on the full scale;
- **ternary** — labels collapsed to True / Mixed / False;
- **binary** — collapsed to True / False (Half true counts as True).

A prediction of `"Not enough information"` is never correct here (every gold
label is a real verdict), but its *rate* is worth watching: it is the model
saying "I can't verify this" — arguably the honest closed-book answer.


In [ ]:
records = []
for condition, verdicts in [("closed_book", closed_book), ("open_book", open_book)]:
    for row, verdict in zip(sample_df.itertuples(), verdicts):
        if verdict is None:
            continue
        records.append({
            "factcheck_analysis_link": row.factcheck_analysis_link,
            "statement": row.statement,
            "condition": condition,
            "gold": row.verdict,
            "predicted": verdict.label,
            "justification": verdict.justification,
        })

results_df = pd.DataFrame(records)
results_df.to_csv("data/nb01_results.csv", index=False)
print("Saved per-claim results -> data/nb01_results.csv\n")


def scenario_accuracy(df):
    scores = {}
    for name, (label_map, _order) in SCENARIOS.items():
        if label_map is None:
            gold, pred = df["gold"], df["predicted"]
        else:
            gold, pred = df["gold"].map(label_map), df["predicted"].map(label_map)
        scores[name] = (gold == pred).mean()
    scores["nei_rate"] = (df["predicted"] == NEI).mean()
    scores["n"] = len(df)
    return pd.Series(scores)


summary = pd.DataFrame(
    {condition: scenario_accuracy(group) for condition, group in results_df.groupby("condition")}
).T.round(3)
summary


In [ ]:
# Where did web search change the verdict?
pivot = results_df.pivot_table(
    index="statement", columns="condition", values="predicted", aggfunc="first"
)
pivot["gold"] = sample_df.set_index("statement")["verdict"]
changed = pivot[pivot["closed_book"] != pivot["open_book"]]
print(f"Web search changed the verdict on {len(changed)} of {len(pivot)} claims:")
changed


## Takeaways

- **Closed-book fact-checking is a memory test.** For claims outside the
  model's training window or media spotlight, the honest answer is
  *Not enough information* — watch how often the model says so versus
  guessing a verdict.
- **Web search helps, but is not a fix.** Uncurated retrieval frequently
  fails to confirm complex political claims (DeVerna, ACL 2026) — compare the
  ternary/binary gains against what you might have expected.
- **Granularity matters.** The multi-class vs. ternary vs. binary gaps show
  how much "error" is rating-scale disagreement rather than factual error.

**Next:** Notebook 2 acquires genuinely *new* information — today's news —
so we can audit real-time knowledge on facts no model has trained on.

> Results are saved to `data/nb01_results.csv`; Notebook 5 reuses both
> conditions as single-model baselines for the multi-agent debate comparison.
